# Khởi tạo Môi trường Google Colab T4 & Đồng bộ Kho Lưu trữ

Cell bên dưới sẽ tự động thực hiện:
1. **Kết nối Google Drive** và tạo các thư mục lưu trữ (`feature_store`, `checkpoints`).
2. **Kiểm tra thư mục code cũ**: Nếu đã tồn tại thì tự động xóa sạch và clone bản mới nhất từ GitHub.
3. **Chuyển vào thư mục làm việc chuẩn** và đăng ký `sys.path` để nạp module `benchmarks` không bao giờ bị lỗi.
4. **Kiểm tra phần cứng GPU T4** và xác thực sự hiện diện của các thư mục `core/`, `models/`, `metrics/`.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# -------------------------------------------------------------------------
# 1. Kết nối Google Drive & Khởi tạo Thư mục Lưu trữ
# -------------------------------------------------------------------------
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print('[1/4] Đang kết nối Google Drive...')
        drive.mount('/content/drive')
    else:
        print('[1/4] Google Drive đã được kết nối!')
except ImportError:
    print('[1/4] Chạy trên môi trường Local (Bỏ qua Colab Drive Mount)')

DRIVE_DIR = '/content/drive/MyDrive/multimodal_lecture_benchmark'
os.makedirs(f'{DRIVE_DIR}/feature_store', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/hf_models', exist_ok=True)
print(f'[OK] Thư mục lưu trữ Drive: {DRIVE_DIR}')

# -------------------------------------------------------------------------
# 2. Kiểm tra, Xóa bản cũ và Clone bản mới nhất từ GitHub
# -------------------------------------------------------------------------
REPO_URL = 'https://github.com/multimodal-lecture-summarizer/multimodal-lecture-summarizer.git'
TARGET_ROOT = '/content/multimodal-lecture-summarizer'

%cd /content
if os.path.exists(TARGET_ROOT):
    print(f'[2/4] Phát hiện thư mục code cũ tại {TARGET_ROOT}, đang xóa sạch...')
    shutil.rmtree(TARGET_ROOT)
    print('[OK] Đã xóa bản cũ thành công.')

print(f'[2/4] Đang clone mã nguồn mới nhất từ GitHub...')
!git clone {REPO_URL} {TARGET_ROOT}

# -------------------------------------------------------------------------
# 3. Chuyển vào đúng thư mục dự án và Đăng ký sys.path
# -------------------------------------------------------------------------
PROJECT_DIR = f'{TARGET_ROOT}'
%cd {PROJECT_DIR}

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Tránh xung đột thư viện OpenMP
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# HF_HOME trỏ vào Drive để tái dùng model đã download giữa các session
HF_HOME_DRIVE = f'{DRIVE_DIR}/hf_models'
os.environ['HF_HOME'] = HF_HOME_DRIVE
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_HOME_DRIVE
os.environ['SENTENCE_TRANSFORMERS_HOME'] = f'{HF_HOME_DRIVE}/sentence_transformers'
print(f'[OK] HF_HOME = {HF_HOME_DRIVE} (Drive-backed, tái dùng giữa sessions)')

# -------------------------------------------------------------------------
# 3b. Pre-download các model cần thiết vào Drive (chỉ download lần đầu)
#     Sau đó set TRANSFORMERS_OFFLINE=1 để tắt hoàn toàn HF network calls
#     → tránh HTTP 429 rate-limit khi chạy nhiều lần
# -------------------------------------------------------------------------
MODELS_TO_CACHE = [
    ('sentence-transformers/all-MiniLM-L6-v2', 'sentence_transformers'),
]

for model_id, model_type in MODELS_TO_CACHE:
    model_local = f'{HF_HOME_DRIVE}/sentence_transformers/{model_id.replace("/", "_")}'
    if os.path.exists(model_local) and os.listdir(model_local):
        print(f'[Cache HIT] {model_id} — đã có trong Drive, bỏ qua download')
    else:
        print(f'[Cache MISS] Đang download {model_id} vào Drive (1 lần duy nhất)...')
        try:
            from sentence_transformers import SentenceTransformer
            _m = SentenceTransformer(model_id, device='cpu', cache_folder=f'{HF_HOME_DRIVE}/sentence_transformers')
            del _m
            print(f'[OK] {model_id} đã lưu vào Drive.')
        except Exception as _e:
            print(f'[WARN] Download {model_id} thất bại: {_e} — sẽ dùng keyword fallback.')

# Tắt HF network sau khi đã có model local — ngăn hoàn toàn 429
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
print('[OK] TRANSFORMERS_OFFLINE=1 — mọi HF call sẽ dùng cache local, không gọi mạng.')

# -------------------------------------------------------------------------
# 4. Chẩn đoán Môi trường & Xác thực Cấu trúc benchmarks/
# -------------------------------------------------------------------------
print('\n' + '='*60)
print('[OK] HOÀN TẤT THIẾT LẬP MÔI TRƯỜNG!')
print(f'- Thư mục hiện tại (CWD): {os.getcwd()}')

benchmarks_path = Path(PROJECT_DIR) / 'benchmarks'
if benchmarks_path.exists():
    subdirs = [d.name for d in benchmarks_path.iterdir() if d.is_dir()]
    print(f'- Module benchmarks/ đã sẵn sàng: {subdirs}')
else:
    print('[CẢNH BÁO] Không tìm thấy thư mục benchmarks/!')

import torch
gpu_status = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Chỉ có CPU (Chưa bật GPU T4)'
print(f'- Phần cứng: {gpu_status}')
print('='*60)
print('Bây giờ bạn có thể mở và chạy bất kỳ notebook nào (01, 02, 03, 04, 05)!')
